<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/skullstriping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kafatası Sıyırma (Skull Stripping / Brain Extraction)
MRI görüntülerinde beyin dokusunun dışındaki yapılar (kafatası, deri, yağ, gözler ve kaslar) Alzheimer teşhisi için kullanılan öznitelik çıkarımı ve hacimsel analiz süreçlerini olumsuz etkileyebilir. Skull Stripping işlemi, görüntüyü sadece beyin dokusunu (gri madde, beyaz madde ve beyin omurilik sıvısı) içerecek şekilde izole ederek analizlerin doğruluğunu artırır.
# Neden HD-BET Tercih Edildi?
Kafatası sıyırma işlemi için geleneksel yöntemler yerine yapay zeka tabanlı HD-BET'in seçilme nedeni, derin öğrenme (nnU-Net) mimarisi sayesinde beyin sınırlarını çok daha keskin ve hatasız belirleyebilmesidir. Alzheimer kaynaklı beyin küçülmelerinde (atrofi) dahi anatomik varyasyonlardan etkilenmeden kararlı sonuçlar üreten bu araç, A100 GPU donanımını kullanarak büyük veri setlerini yüksek hızda işleyebilmektedir. Ayrıca, analiz aşamalarında kritik rol oynayan ikili beyin maskelerini (_bet.nii.gz) otomatik olarak oluşturarak veri işleme hattının sürekliliğini sağlamaktadır.

In [ ]:
# HD-BET Kurulumu [cite: 2026-03-05]
!git clone https://github.com/MIC-DKFZ/HD-BET
%cd HD-BET
!pip install -e .
%cd ..

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)}")
else:
    print("GPU bulunamadı!")

In [ ]:
CONFIG_CN = {
    "kategori": "CN",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/n4_bias/n4_CN',
    "hedef": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_stripingg/skull_striping_CN'
}


In [ ]:
CONFIG_EMCI = {
    "kategori": "EMCI",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/n4_bias/n4_EMCI',
    "hedef": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_stripingg/skull_striping_EMCI'
}

In [ ]:
import os
import shutil
import subprocess

def hd_bet_uygula(config, batch_size=20):
    source_base = config["kaynak"]
    output_base = config["hedef"]
    etiket = config["kategori"]

    local_in = "/content/batch_in_hdbet"
    local_out = "/content/batch_out_hdbet"

    print(f"\n {etiket} Kafatası sıyırma  işlemi başlıyor (Grup Boyutu: {batch_size})")


    all_pending = []
    subjects = sorted([s for s in os.listdir(source_base) if os.path.isdir(os.path.join(source_base, s))])

    toplam_dosya = 0
    atlanan_dosya = 0
    basarili_dosya = 0
    hatali_dosya = 0

    print(" Klasörler taranıyor, lütfen bekleyin.", end="\r")

    for subject_id in subjects:
        subject_path = os.path.join(source_base, subject_id)
        seanslar = [d for d in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, d))]

        for seans in seanslar:
            seans_yolu = os.path.join(subject_path, seans)
            nifti_files = [f for f in os.listdir(seans_yolu) if f.endswith('.nii.gz')]

            if nifti_files:
                toplam_dosya += 1
                f = nifti_files[0]
                rel_path = os.path.join(subject_id, seans)
                cikti_yolu = os.path.join(output_base, rel_path, f)

                if not os.path.exists(cikti_yolu):
                    all_pending.append((os.path.join(seans_yolu, f), rel_path, f, subject_id))
                else:
                    atlanan_dosya += 1

    #Kafatası sıyırma işlemine başlamadan önce klaösrde toplam kaç dosya bulunduğu,kaçının dönüştürüldüğü,o an kaç dosyanın dönüştürüldüğünü özetler.
    kalan_dosya = len(all_pending)
    print(" "*50, end="\r")
    print(f" KLASÖR ANALİZİ ({etiket}):")
    print(f" Toplam Bulunan Dosya : {toplam_dosya}")
    print(f" Zaten İşlenmiş Olan  : {atlanan_dosya}")
    print(f" Şimdi İşlenecek Olan : {kalan_dosya}")
    print("-" * 40)
    # ---------------------------------------------

    if not all_pending:
        print(f" İşlenecek yeni dosya yok. Sistem durduruluyor.")
    else:

        for i in range(0, kalan_dosya, batch_size):
            batch = all_pending[i:i+batch_size]
            shutil.rmtree(local_in, ignore_errors=True)
            shutil.rmtree(local_out, ignore_errors=True)
            os.makedirs(local_in, exist_ok=True)
            os.makedirs(local_out, exist_ok=True)

            batch_subjects = sorted(list(set([item[3] for item in batch])))
            print(f"\n Grup {i//batch_size + 1} Başlıyor...")
            print(f" İşlenen Hastalar: {', '.join(batch_subjects)}")

            mapping = {}
            for full_src, rel_p, fname, sid in batch:
                unique_name = rel_p.replace("/", "_") + "___" + fname
                shutil.copy2(full_src, os.path.join(local_in, unique_name))
                mapping[unique_name] = (rel_p, fname)

            print(f" Kafatası sıyırma işlemi çalışıyor. ", end="", flush=True)
            cmd = f'hd-bet -i "{local_in}" -o "{local_out}" -device cuda:0 --disable_tta --save_bet_mask' #disable_tta ile beyinin sürekli döndürülmesi engellenerek zaman kazanılmıştır.save_bet_mask ile kullanılan maskelerinde drive kayda yapıldı daha sonra kullanmak için.

            try:
                subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
                print(" Bitti")

                print(f" Drive'a aktarılıyor. ", end="", flush=True)
                for unique_name, (rel_p, fname) in mapping.items():
                    l_res = os.path.join(local_out, unique_name)
                    l_mask = os.path.join(local_out, unique_name.replace(".nii.gz", "_bet.nii.gz"))
                    d_target_dir = os.path.join(output_base, rel_p)
                    os.makedirs(d_target_dir, exist_ok=True)

                    if os.path.exists(l_res):
                        shutil.move(l_res, os.path.join(d_target_dir, fname))
                        basarili_dosya += 1
                    if os.path.exists(l_mask):
                        shutil.move(l_mask, os.path.join(d_target_dir, fname.replace(".nii.gz", "_bet.nii.gz")))


                su_an_biten = min(i+batch_size, kalan_dosya)
                print(f" {su_an_biten}/{kalan_dosya} yeni dosya kaydedildi.")

            except Exception as e:
                print(f" Grup Hatası: {str(e)}")
                hatali_dosya += len(batch)

    # İşlem bittiğinde neler olduğuna dair bir özet
    print(f"\n{etiket} HD-BET Özeti:")
    print(f"Toplam   : {toplam_dosya}")
    print(f"Başarılı : {basarili_dosya}")
    print(f"Atlanan  : {atlanan_dosya}")
    print(f"Hatalı   : {hatali_dosya}")
    print(f"{etiket} grubu tamamlandı.")

In [ ]:

hd_bet_uygula(CONFIG_CN)

In [ ]:
hd_bet_uygula(CONFIG_EMCI)

In [ ]:
import os

def klasor_analiz(config):
    hedef = config["hedef"]
    etiket = config["kategori"]

    print(f" {etiket} Klasör Analizi: {hedef}")

    all_nii_files = []
    all_bet_files = []
    unique_patient_ids = set()


    for root, dirs, files in os.walk(hedef):

        if os.path.basename(root) != etiket and os.path.basename(os.path.dirname(root)) != etiket:
            parts = root.split(os.sep)

            try:
                base_index = parts.index(os.path.basename(hedef))
                if len(parts) > base_index + 1:
                    patient_id = parts[base_index + 1]
                    unique_patient_ids.add(patient_id)
            except ValueError:

                pass

        for file in files:
            if file.endswith('.nii.gz'):
                full_path = os.path.join(root, file)
                if '_bet.nii.gz' in file:
                    all_bet_files.append(full_path)
                else:
                    all_nii_files.append(full_path)

    print(f"   Toplam farklı hasta     : {len(unique_patient_ids)}")
    print(f"   Toplam '.nii.gz' dosyası : {len(all_nii_files) + len(all_bet_files)}")
    print(f"   Beyin görüntüsü         : {len(all_nii_files)}  (kafatası sıyrılmış .nii.gz)")
    print(f"   Maske ('_bet.nii.gz')   : {len(all_bet_files)}  (ikili beyin maskesi)")

    eksik_maske = []
    for nii_file_path in all_nii_files:

        filename_only = os.path.basename(nii_file_path).replace('.nii.gz', '')
        expected_mask_filename = filename_only + '_bet.nii.gz'
        expected_mask_path = os.path.join(os.path.dirname(nii_file_path), expected_mask_filename)

        if expected_mask_path not in all_bet_files:
            eksik_maske.append(os.path.basename(nii_file_path))

    if eksik_maske:
        print(f"  Maskesi eksik olan beyin görüntüleri ({len(eksik_maske)}):")
        if len(eksik_maske) > 5:
            for f in eksik_maske[:5]:
                print(f"      - {f}")
            print(f"      ...ve {len(eksik_maske) - 5} diğer dosya.")
        else:
            for f in eksik_maske:
                print(f"      - {f}")
    else:
        print(f"  Her beyin görüntüsünün maskesi mevcut.")

    print("="*80)

In [ ]:
klasor_analiz(CONFIG_CN)


In [ ]:
klasor_analiz(CONFIG_EMCI)